## Metrics

In [259]:
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, hamming_loss, multilabel_confusion_matrix, classification_report
import langcodes
from tqdm import tqdm
from flores201_classes import flores_201_langs

In [260]:
def calculate_exact_match_ratio(bin_gold_labels, bin_pred_labels):
    return accuracy_score(bin_gold_labels, bin_pred_labels)

In [261]:
def calculate_hamming_loss(bin_gold_labels, bin_pred_labels):
    return hamming_loss(bin_gold_labels, bin_pred_labels)

In [262]:
def calculate_false_positive_rate(bin_gold_labels, bin_pred_labels):
    mcm = multilabel_confusion_matrix(bin_gold_labels, bin_pred_labels, samplewise=False)
    tn = mcm[:, 0, 0]
    tp = mcm[:, 1, 1]
    fn = mcm[:, 1, 0]
    fp = mcm[:, 0, 1]
    fpr = np.nan_to_num(fp/(fp+tn)) 
    return fpr.mean()

In [263]:
def calculate_classification_report(bin_gold_labels, bin_pred_labels, mlb):
    report = classification_report(bin_gold_labels, 
                                   bin_pred_labels, 
                                   target_names=mlb.classes_, 
                                   zero_division=0, 
                                   digits=3)
    return report

In [264]:
def calculate_num_unique_langs_predicted(pred_labels):
    return len(set([item for sublist in pred_labels for item in sublist]))

## OpenLID

In [265]:
openlid_results = pd.read_csv("/usr/local/data/zkamel/COMP598_Project/dataset/code-switch/codeswitch_baselines_results/codeswitch/OpenLID_multi_cs.csv")

In [266]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594
...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008


In [267]:
unique_languages = openlid_results['languages'].unique()

In [268]:
unique_languages

array(['English-Tamil', 'Basque-Spanish', 'English-Hindi',
       'Chinese-English', 'English-Spanish', 'Egyptian-English',
       'Egyptian-MSA', 'English-Malayalam', 'English-Saudi',
       'English-German', 'English-Turkish', 'Arabizi-English',
       'English-Indonesian'], dtype=object)

In [269]:
unique_langs = set()
for label in unique_languages:
    langs = label.split('-')
    unique_langs.update(langs)

unique_langs = list(unique_langs)
unique_langs.sort()

print(unique_langs)

['Arabizi', 'Basque', 'Chinese', 'Egyptian', 'English', 'German', 'Hindi', 'Indonesian', 'MSA', 'Malayalam', 'Saudi', 'Spanish', 'Tamil', 'Turkish']


In [270]:
language_to_iso_script = {
    'Arabizi': 'arb_Latn',
    'Basque': 'eus_Latn',
    'Chinese': 'zho_Hans', 
    'Egyptian': 'arz_Arab',
    'English': 'eng_Latn',
    'German': 'deu_Latn',
    'Hindi': 'hin_Deva',
    'Indonesian': 'ind_Latn',
    'MSA': 'arb_Arab',       
    'Malayalam': 'mal_Mlym',
    'Saudi': 'ars_Arab',
    'Spanish': 'spa_Latn',
    'Tamil': 'tam_Taml',
    'Turkish': 'tur_Latn'
}

In [271]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [272]:
openlid_results['iso_pair'] = openlid_results['languages'].apply(map_to_iso_pair)

In [273]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893,"[eng_Latn, tam_Taml]"
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932,"[eus_Latn, spa_Latn]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937,"[eng_Latn, hin_Deva]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608,"[eng_Latn, tam_Taml]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594,"[zho_Hans, eng_Latn]"
...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585,"[eng_Latn, hin_Deva]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087,"[eng_Latn, tam_Taml]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591,"[zho_Hans, eng_Latn]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008,"[arz_Arab, eng_Latn]"


In [274]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [275]:
openlid_results['top_pred']

0        {'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...
1        {'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...
2        {'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...
3        {'yor_Latn': 0.19660769402980804, 'hau_Latn': ...
4        {'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...
                               ...                        
35609    {'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...
35610    {'eng_Latn': 0.30808666348457336, 'est_Latn': ...
35611    {'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...
35612    {'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...
35613    {'eng_Latn': 0.881960928440094, 'pol_Latn': 0....
Name: top_pred, Length: 35614, dtype: object

In [276]:
import ast

openlid_results["predicted_top_2"] = openlid_results["top_pred"].apply(
    lambda s: list(ast.literal_eval(s).keys())[:2]
)

In [277]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893,"[eng_Latn, tam_Taml]","[zho_Hant, zho_Hans]"
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932,"[eus_Latn, spa_Latn]","[eus_Latn, hrv_Latn]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937,"[eng_Latn, hin_Deva]","[eng_Latn, swh_Latn]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608,"[eng_Latn, tam_Taml]","[yor_Latn, hau_Latn]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594,"[zho_Hans, eng_Latn]","[zho_Hans, zho_Hant]"
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585,"[eng_Latn, hin_Deva]","[eng_Latn, dan_Latn]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087,"[eng_Latn, tam_Taml]","[eng_Latn, est_Latn]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591,"[zho_Hans, eng_Latn]","[zho_Hans, zho_Hant]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008,"[arz_Arab, eng_Latn]","[arz_Arab, ajp_Arab]"


In [278]:
def convert_to_iso_pair(language_pair):
    lang1, lang2 = language_pair.split('-')
    iso1 = language_to_iso_script[lang1]
    iso2 = language_to_iso_script[lang2]
    iso_pair = f"{iso1.split('_')[0]}-{iso2.split('_')[0]}"
    return iso_pair

languages = ['English-Tamil', 'Basque-Spanish', 'English-Hindi',
             'Chinese-English', 'English-Spanish', 'Egyptian-English',
             'Egyptian-MSA', 'English-Malayalam', 'English-Saudi',
             'English-German', 'English-Turkish', 'Arabizi-English',
             'English-Indonesian']

iso_pairs = [convert_to_iso_pair(pair) for pair in languages]

print(iso_pairs)

['eng-tam', 'eus-spa', 'eng-hin', 'zho-eng', 'eng-spa', 'arz-eng', 'arz-arb', 'eng-mal', 'eng-ars', 'eng-deu', 'eng-tur', 'arb-eng', 'eng-ind']


In [279]:
import pandas as pd

target_groups = [
    'eng_Latn-tam_Taml', 'eus_Latn-spa_Latn', 'eng_Latn-hin_Deva', 'zho_Hans-eng_Latn', 'eng_Latn-spa_Latn', 'arz_Arab-eng_Latn',
    'arz_Arab-arb_Arab', 'eng_Latn-mal_Mlym', 'eng_Latn-ars_Arab', 'eng_Latn-deu_Latn', 'eng_Latn-tur_Latn', 'arb_Latn-eng_Latn', 'eng_Latn-ind_Latn'
]

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

openlid_results['match_type'], openlid_results['correct_lang'] = zip(*openlid_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = openlid_results[openlid_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match': full_match_count,
        'Partial Match': partial_match_count,
        'No Match': no_match_count,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T
print(results_df)


                  Full Match Partial Match No Match  \
eng_Latn-tam_Taml          1          1135     3134   
eus_Latn-spa_Latn         66           334       12   
eng_Latn-hin_Deva          0          2597     4844   
zho_Hans-eng_Latn        101          6344      118   
eng_Latn-spa_Latn         43          4686     2068   
arz_Arab-eng_Latn         66          2608      266   
arz_Arab-arb_Arab         96          3277      745   
eng_Latn-mal_Mlym          5           254     1451   
eng_Latn-ars_Arab          7           336       84   
eng_Latn-deu_Latn         11           241        1   
eng_Latn-tur_Latn          5           110        3   
arb_Latn-eng_Latn          0            70      247   
eng_Latn-ind_Latn         23           198       27   

                  Most Frequent Correct Language (Partial)  
eng_Latn-tam_Taml                                 eng_Latn  
eus_Latn-spa_Latn                                 eus_Latn  
eng_Latn-hin_Deva                             

In [280]:
def get_target_group(iso_pair, target_groups):
    if isinstance(iso_pair, list):
        iso_pair = '-'.join(iso_pair)  
    
    for target_group in target_groups:
        if set(iso_pair.split('-')) == set(target_group.split('-')):
            return target_group
    return None 

openlid_results['target_group'] = openlid_results['iso_pair'].apply(
    lambda iso_pair: get_target_group(iso_pair, target_groups)
)


In [281]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2,match_type,correct_lang,target_group
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893,"[eng_Latn, tam_Taml]","[zho_Hant, zho_Hans]",No Match,None,eng_Latn-tam_Taml
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932,"[eus_Latn, spa_Latn]","[eus_Latn, hrv_Latn]",Partial Match,eus_Latn,eus_Latn-spa_Latn
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937,"[eng_Latn, hin_Deva]","[eng_Latn, swh_Latn]",Partial Match,eng_Latn,eng_Latn-hin_Deva
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608,"[eng_Latn, tam_Taml]","[yor_Latn, hau_Latn]",No Match,None,eng_Latn-tam_Taml
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594,"[zho_Hans, eng_Latn]","[zho_Hans, zho_Hant]",Partial Match,zho_Hans,zho_Hans-eng_Latn
...,...,...,...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585,"[eng_Latn, hin_Deva]","[eng_Latn, dan_Latn]",Partial Match,eng_Latn,eng_Latn-hin_Deva
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087,"[eng_Latn, tam_Taml]","[eng_Latn, est_Latn]",Partial Match,eng_Latn,eng_Latn-tam_Taml
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591,"[zho_Hans, eng_Latn]","[zho_Hans, zho_Hant]",Partial Match,zho_Hans,zho_Hans-eng_Latn
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008,"[arz_Arab, eng_Latn]","[arz_Arab, ajp_Arab]",Partial Match,arz_Arab,arz_Arab-eng_Latn


In [282]:
import pandas as pd

target_groups = [
    'eng_Latn-tam_Taml', 'eus_Latn-spa_Latn', 'eng_Latn-hin_Deva', 'zho_Hans-eng_Latn', 'eng_Latn-spa_Latn', 'arz_Arab-eng_Latn',
    'arz_Arab-arb_Arab', 'eng_Latn-mal_Mlym', 'eng_Latn-ars_Arab', 'eng_Latn-deu_Latn', 'eng_Latn-tur_Latn', 'arb_Latn-eng_Latn', 'eng_Latn-ind_Latn'
]

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

openlid_results['match_type'], openlid_results['correct_lang'] = zip(*openlid_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = openlid_results[openlid_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    total_count = len(target_group_rows)
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match (%)': (full_match_count / total_count) * 100,
        'Partial Match (%)': (partial_match_count / total_count) * 100,
        'No Match (%)': (no_match_count / total_count) * 100,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T

def calculate_exact_match(true_label, predicted_label):
    return 1 if set(true_label) == set(predicted_label) else 0

def calculate_hamming_loss(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    mismatches = len(true_label_set.symmetric_difference(pred_label_set))
    return mismatches / len(true_label_set) if len(true_label_set) > 0 else 0

def calculate_fpr(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    false_positives = len(pred_label_set - true_label_set)
    return false_positives / len(true_label_set) if len(true_label_set) > 0 else 0

openlid_results['exact_match'] = openlid_results.apply(
    lambda row: calculate_exact_match(row['iso_pair'], row['predicted_top_2']), axis=1)

openlid_results['hamming_loss'] = openlid_results.apply(
    lambda row: calculate_hamming_loss(row['iso_pair'], row['predicted_top_2']), axis=1)

openlid_results['fpr'] = openlid_results.apply(
    lambda row: calculate_fpr(row['iso_pair'], row['predicted_top_2']), axis=1)

grouped_results = openlid_results.groupby('target_group').agg(
    exact_match=('exact_match', 'mean'),
    hamming_loss=('hamming_loss', 'mean'),
    fpr=('fpr', 'mean'),
).reset_index()

grouped_results_openlid = grouped_results.merge(results_df, left_on='target_group', right_index=True)

print(grouped_results_openlid)


         target_group  exact_match  hamming_loss       fpr Full Match (%)  \
0   arb_Latn-eng_Latn     0.000000      1.779180  0.889590            0.0   
1   arz_Arab-arb_Arab     0.023312      1.157601  0.578800       2.331229   
2   arz_Arab-eng_Latn     0.022449      1.068027  0.534014       2.244898   
3   eng_Latn-ars_Arab     0.016393      1.180328  0.590164       1.639344   
4   eng_Latn-deu_Latn     0.043478      0.960474  0.480237       4.347826   
5   eng_Latn-hin_Deva     0.000000      1.650988  0.825494            0.0   
6   eng_Latn-ind_Latn     0.092742      1.016129  0.508065       9.274194   
7   eng_Latn-mal_Mlym     0.002924      1.845614  0.922807       0.292398   
8   eng_Latn-spa_Latn     0.006326      1.297926  0.648963       0.632632   
9   eng_Latn-tam_Taml     0.000234      1.733724  0.866862       0.023419   
10  eng_Latn-tur_Latn     0.042373      0.983051  0.491525       4.237288   
11  eus_Latn-spa_Latn     0.160194      0.868932  0.434466      16.019417   

## Franc

In [283]:
franc_results = pd.read_csv("/usr/local/data/zkamel/COMP598_Project/dataset/code-switch/codeswitch_baselines_results/codeswitch/franc_multi_cs.csv")

In [284]:
franc_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob
0,Yellam Avan seyal . . .,English-Tamil,"{'fao': 1.0, 'hun': 0.9845837615621789, 'mos':...",fao,1.0
1,zein da kanala!,Basque-Spanish,"{'emk': 1.0, 'bam': 0.9340620592383639, 'war':...",emk,1.0
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng': 1.0, 'sco': 0.9708080780625696, 'pcm':...",eng,1.0
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'buc': 1.0, 'kal': 0.9903674634320371, 'mxv':...",buc,1.0
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,{'cmn': 1.0},cmn,1.0
...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'sco': 1.0, 'rmn': 0.9833437110834371, 'eng':...",sco,1.0
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'afr': 1.0, 'kng': 0.9945155393053017, 'cha':...",afr,1.0
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,{'cmn': 1.0},cmn,1.0
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arb': 1.0, 'zlm': 0.8602086140639816, 'pes':...",arb,1.0


In [285]:
language_to_iso_script = {
    'Arabizi': 'arb',
    'Basque': 'eus',
    'Chinese': 'zho', 
    'Egyptian': 'arz',
    'English': 'eng',
    'German': 'deu',
    'Hindi': 'hin',
    'Indonesian': 'ind',
    'MSA': 'arb',       
    'Malayalam': 'mal',
    'Saudi': 'ars',
    'Spanish': 'spa',
    'Tamil': 'tam',
    'Turkish': 'tur'
}

In [286]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [287]:
franc_results['iso_pair'] = franc_results['languages'].apply(map_to_iso_pair)

In [288]:
franc_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair
0,Yellam Avan seyal . . .,English-Tamil,"{'fao': 1.0, 'hun': 0.9845837615621789, 'mos':...",fao,1.0,"[eng, tam]"
1,zein da kanala!,Basque-Spanish,"{'emk': 1.0, 'bam': 0.9340620592383639, 'war':...",emk,1.0,"[eus, spa]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng': 1.0, 'sco': 0.9708080780625696, 'pcm':...",eng,1.0,"[eng, hin]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'buc': 1.0, 'kal': 0.9903674634320371, 'mxv':...",buc,1.0,"[eng, tam]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]"
...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'sco': 1.0, 'rmn': 0.9833437110834371, 'eng':...",sco,1.0,"[eng, hin]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'afr': 1.0, 'kng': 0.9945155393053017, 'cha':...",afr,1.0,"[eng, tam]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arb': 1.0, 'zlm': 0.8602086140639816, 'pes':...",arb,1.0,"[arz, eng]"


In [289]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [290]:
import ast

franc_results["predicted_top_2"] = franc_results["top_pred"].apply(
    lambda s: list(ast.literal_eval(s).keys())[:2]
)

In [291]:
franc_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,"{'fao': 1.0, 'hun': 0.9845837615621789, 'mos':...",fao,1.0,"[eng, tam]","[fao, hun]"
1,zein da kanala!,Basque-Spanish,"{'emk': 1.0, 'bam': 0.9340620592383639, 'war':...",emk,1.0,"[eus, spa]","[emk, bam]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng': 1.0, 'sco': 0.9708080780625696, 'pcm':...",eng,1.0,"[eng, hin]","[eng, sco]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'buc': 1.0, 'kal': 0.9903674634320371, 'mxv':...",buc,1.0,"[eng, tam]","[buc, kal]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]",[cmn]
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'sco': 1.0, 'rmn': 0.9833437110834371, 'eng':...",sco,1.0,"[eng, hin]","[sco, rmn]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'afr': 1.0, 'kng': 0.9945155393053017, 'cha':...",afr,1.0,"[eng, tam]","[afr, kng]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]",[cmn]
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arb': 1.0, 'zlm': 0.8602086140639816, 'pes':...",arb,1.0,"[arz, eng]","[arb, zlm]"


In [292]:
def convert_to_iso_pair(language_pair):
    lang1, lang2 = language_pair.split('-')
    iso1 = language_to_iso_script[lang1]
    iso2 = language_to_iso_script[lang2]
    iso_pair = f"{iso1.split('_')[0]}-{iso2.split('_')[0]}"
    return iso_pair

languages = ['English-Tamil', 'Basque-Spanish', 'English-Hindi',
             'Chinese-English', 'English-Spanish', 'Egyptian-English',
             'Egyptian-MSA', 'English-Malayalam', 'English-Saudi',
             'English-German', 'English-Turkish', 'Arabizi-English',
             'English-Indonesian']

iso_pairs = [convert_to_iso_pair(pair) for pair in languages]

print(iso_pairs)

['eng-tam', 'eus-spa', 'eng-hin', 'zho-eng', 'eng-spa', 'arz-eng', 'arz-arb', 'eng-mal', 'eng-ars', 'eng-deu', 'eng-tur', 'arb-eng', 'eng-ind']


In [293]:
import pandas as pd

target_groups = ['eng-tam', 'eus-spa', 'eng-hin', 'zho-eng', 'eng-spa', 'arz-eng', 'arz-arb', 'eng-mal', 'eng-ars', 'eng-deu', 'eng-tur', 'arb-eng', 'eng-ind']

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

franc_results['match_type'], franc_results['correct_lang'] = zip(*franc_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = franc_results[franc_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match': full_match_count,
        'Partial Match': partial_match_count,
        'No Match': no_match_count,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T
print(results_df)


        Full Match Partial Match No Match  \
eng-tam          0           123     4147   
eus-spa          3           222      187   
eng-hin          0          2023     5418   
zho-eng          0           255     6308   
eng-spa          9          3260     3528   
arz-eng          0           226     2714   
arz-arb          0          3216      902   
eng-mal          0            40     1670   
eng-ars          0            26      401   
eng-deu          3           244        6   
eng-tur          0            77       41   
arb-eng          0            83      234   
eng-ind          2           124      122   

        Most Frequent Correct Language (Partial)  
eng-tam                                      eng  
eus-spa                                      eus  
eng-hin                                      eng  
zho-eng                                      eng  
eng-spa                                      spa  
arz-eng                                      eng  
arz-arb     

In [294]:
def get_target_group(iso_pair, target_groups):
    if isinstance(iso_pair, list):
        iso_pair = '-'.join(iso_pair)  
    
    for target_group in target_groups:
        if set(iso_pair.split('-')) == set(target_group.split('-')):
            return target_group
    return None 

franc_results['target_group'] = franc_results['iso_pair'].apply(
    lambda iso_pair: get_target_group(iso_pair, target_groups)
)


In [295]:
franc_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2,match_type,correct_lang,target_group
0,Yellam Avan seyal . . .,English-Tamil,"{'fao': 1.0, 'hun': 0.9845837615621789, 'mos':...",fao,1.0,"[eng, tam]","[fao, hun]",No Match,None,eng-tam
1,zein da kanala!,Basque-Spanish,"{'emk': 1.0, 'bam': 0.9340620592383639, 'war':...",emk,1.0,"[eus, spa]","[emk, bam]",No Match,None,eus-spa
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng': 1.0, 'sco': 0.9708080780625696, 'pcm':...",eng,1.0,"[eng, hin]","[eng, sco]",Partial Match,eng,eng-hin
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'buc': 1.0, 'kal': 0.9903674634320371, 'mxv':...",buc,1.0,"[eng, tam]","[buc, kal]",No Match,None,eng-tam
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]",[cmn],No Match,None,zho-eng
...,...,...,...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'sco': 1.0, 'rmn': 0.9833437110834371, 'eng':...",sco,1.0,"[eng, hin]","[sco, rmn]",No Match,None,eng-hin
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'afr': 1.0, 'kng': 0.9945155393053017, 'cha':...",afr,1.0,"[eng, tam]","[afr, kng]",No Match,None,eng-tam
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]",[cmn],No Match,None,zho-eng
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arb': 1.0, 'zlm': 0.8602086140639816, 'pes':...",arb,1.0,"[arz, eng]","[arb, zlm]",No Match,None,arz-eng


In [296]:
import pandas as pd

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

franc_results['match_type'], franc_results['correct_lang'] = zip(*franc_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = franc_results[franc_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    total_count = len(target_group_rows)
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match (%)': (full_match_count / total_count) * 100,
        'Partial Match (%)': (partial_match_count / total_count) * 100,
        'No Match (%)': (no_match_count / total_count) * 100,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T

def calculate_exact_match(true_label, predicted_label):
    return 1 if set(true_label) == set(predicted_label) else 0

def calculate_hamming_loss(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    mismatches = len(true_label_set.symmetric_difference(pred_label_set))
    return mismatches / len(true_label_set) if len(true_label_set) > 0 else 0

def calculate_fpr(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    false_positives = len(pred_label_set - true_label_set)
    return false_positives / len(true_label_set) if len(true_label_set) > 0 else 0

franc_results['exact_match'] = franc_results.apply(
    lambda row: calculate_exact_match(row['iso_pair'], row['predicted_top_2']), axis=1)

franc_results['hamming_loss'] = franc_results.apply(
    lambda row: calculate_hamming_loss(row['iso_pair'], row['predicted_top_2']), axis=1)

franc_results['fpr'] = franc_results.apply(
    lambda row: calculate_fpr(row['iso_pair'], row['predicted_top_2']), axis=1)

grouped_results = franc_results.groupby('target_group').agg(
    exact_match=('exact_match', 'mean'),
    hamming_loss=('hamming_loss', 'mean'),
    fpr=('fpr', 'mean'),
).reset_index()

grouped_results_franc = grouped_results.merge(results_df, left_on='target_group', right_index=True)

print(grouped_results_franc)


   target_group  exact_match  hamming_loss       fpr Full Match (%)  \
0       arb-eng     0.000000      1.735016  0.865931            0.0   
1       arz-arb     0.000000      1.149466  0.539947            0.0   
2       arz-eng     0.000000      1.900680  0.939116            0.0   
3       eng-ars     0.000000      1.937939  0.968384            0.0   
4       eng-deu     0.011858      1.011858  0.505929       1.185771   
5       eng-hin     0.000000      1.728128  0.864064            0.0   
6       eng-ind     0.008065      1.483871  0.741935       0.806452   
7       eng-mal     0.000000      1.974269  0.985965            0.0   
8       eng-spa     0.001324      1.516919  0.758055       0.132411   
9       eng-tam     0.000000      1.970726  0.985129            0.0   
10      eng-tur     0.000000      1.347458  0.673729            0.0   
11      eus-spa     0.007282      1.440534  0.717233       0.728155   
12      zho-eng     0.000000      1.510894  0.530321            0.0   

   Pa

## GlotLID

In [297]:
glotlid_results = pd.read_csv("/usr/local/data/zkamel/COMP598_Project/dataset/code-switch/codeswitch_baselines_results/codeswitch/glotlid_multi_cs.csv")

In [298]:
glotlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob
0,Yellam Avan seyal . . .,English-Tamil,"{'tur_Latn': 0.999435544013977, 'kia_Latn': 0....",tur_Latn,0.999436
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.5763978362083435, 'bre_Latn': 0...",eus_Latn,0.576398
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.830464780330658, 'pcm_Latn': 0....",eng_Latn,0.830465
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'rnd_Latn': 0.30418160557746887, 'pol_Latn': ...",rnd_Latn,0.304182
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'cmn_Hani': 0.9767146110534668, 'nan_Hani': 0...",cmn_Hani,0.976715
...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.9581118822097778, 'bzj_Latn': 0...",eng_Latn,0.958112
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.4568457007408142, 'tel_Latn': 0...",eng_Latn,0.456846
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'cmn_Hani': 0.9976520538330078, 'wuu_Hani': 0...",cmn_Hani,0.997652
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 0.9949780106544495, 'ajp_Arab': 0...",arz_Arab,0.994978


In [299]:
language_to_iso_script = {
    'Arabizi': 'arb_Latn',
    'Basque': 'eus_Latn',
    'Chinese': 'zho_Hans', 
    'Egyptian': 'arz_Arab',
    'English': 'eng_Latn',
    'German': 'deu_Latn',
    'Hindi': 'hin_Deva',
    'Indonesian': 'ind_Latn',
    'MSA': 'arb_Arab',       
    'Malayalam': 'mal_Mlym',
    'Saudi': 'ars_Arab',
    'Spanish': 'spa_Latn',
    'Tamil': 'tam_Taml',
    'Turkish': 'tur_Latn'
}

In [300]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [301]:
glotlid_results['iso_pair'] = glotlid_results['languages'].apply(map_to_iso_pair)

In [302]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [303]:
import ast

glotlid_results["predicted_top_2"] = glotlid_results["top_pred"].apply(
    lambda s: list(ast.literal_eval(s).keys())[:2]
)

In [304]:
glotlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,"{'tur_Latn': 0.999435544013977, 'kia_Latn': 0....",tur_Latn,0.999436,"[eng_Latn, tam_Taml]","[tur_Latn, kia_Latn]"
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.5763978362083435, 'bre_Latn': 0...",eus_Latn,0.576398,"[eus_Latn, spa_Latn]","[eus_Latn, bre_Latn]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.830464780330658, 'pcm_Latn': 0....",eng_Latn,0.830465,"[eng_Latn, hin_Deva]","[eng_Latn, pcm_Latn]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'rnd_Latn': 0.30418160557746887, 'pol_Latn': ...",rnd_Latn,0.304182,"[eng_Latn, tam_Taml]","[rnd_Latn, pol_Latn]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'cmn_Hani': 0.9767146110534668, 'nan_Hani': 0...",cmn_Hani,0.976715,"[zho_Hans, eng_Latn]","[cmn_Hani, nan_Hani]"
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.9581118822097778, 'bzj_Latn': 0...",eng_Latn,0.958112,"[eng_Latn, hin_Deva]","[eng_Latn, bzj_Latn]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.4568457007408142, 'tel_Latn': 0...",eng_Latn,0.456846,"[eng_Latn, tam_Taml]","[eng_Latn, tel_Latn]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'cmn_Hani': 0.9976520538330078, 'wuu_Hani': 0...",cmn_Hani,0.997652,"[zho_Hans, eng_Latn]","[cmn_Hani, wuu_Hani]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 0.9949780106544495, 'ajp_Arab': 0...",arz_Arab,0.994978,"[arz_Arab, eng_Latn]","[arz_Arab, ajp_Arab]"


In [305]:
def convert_to_iso_pair(language_pair):
    lang1, lang2 = language_pair.split('-')
    iso1 = language_to_iso_script[lang1]
    iso2 = language_to_iso_script[lang2]
    iso_pair = f"{iso1.split('_')[0]}-{iso2.split('_')[0]}"
    return iso_pair

languages = ['English-Tamil', 'Basque-Spanish', 'English-Hindi',
             'Chinese-English', 'English-Spanish', 'Egyptian-English',
             'Egyptian-MSA', 'English-Malayalam', 'English-Saudi',
             'English-German', 'English-Turkish', 'Arabizi-English',
             'English-Indonesian']

iso_pairs = [convert_to_iso_pair(pair) for pair in languages]

print(iso_pairs)

['eng-tam', 'eus-spa', 'eng-hin', 'zho-eng', 'eng-spa', 'arz-eng', 'arz-arb', 'eng-mal', 'eng-ars', 'eng-deu', 'eng-tur', 'arb-eng', 'eng-ind']


In [306]:
import pandas as pd

target_groups = [
    'eng_Latn-tam_Taml', 'eus_Latn-spa_Latn', 'eng_Latn-hin_Deva', 'zho_Hans-eng_Latn', 'eng_Latn-spa_Latn', 'arz_Arab-eng_Latn',
    'arz_Arab-arb_Arab', 'eng_Latn-mal_Mlym', 'eng_Latn-ars_Arab', 'eng_Latn-deu_Latn', 'eng_Latn-tur_Latn', 'arb_Latn-eng_Latn', 'eng_Latn-ind_Latn'
]

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

glotlid_results['match_type'], glotlid_results['correct_lang'] = zip(*glotlid_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = glotlid_results[glotlid_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match': full_match_count,
        'Partial Match': partial_match_count,
        'No Match': no_match_count,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T
print(results_df)


                  Full Match Partial Match No Match  \
eng_Latn-tam_Taml          2          1269     2999   
eus_Latn-spa_Latn         49           345       18   
eng_Latn-hin_Deva          0          2526     4915   
zho_Hans-eng_Latn          0           390     6173   
eng_Latn-spa_Latn        254          5864      679   
arz_Arab-eng_Latn         32          2582      326   
arz_Arab-arb_Arab        203          3068      847   
eng_Latn-mal_Mlym          5           252     1453   
eng_Latn-ars_Arab          2           314      111   
eng_Latn-deu_Latn         16           234        3   
eng_Latn-tur_Latn          5           108        5   
arb_Latn-eng_Latn          2           123      192   
eng_Latn-ind_Latn         25           193       30   

                  Most Frequent Correct Language (Partial)  
eng_Latn-tam_Taml                                 eng_Latn  
eus_Latn-spa_Latn                                 eus_Latn  
eng_Latn-hin_Deva                             

In [307]:
def get_target_group(iso_pair, target_groups):
    if isinstance(iso_pair, list):
        iso_pair = '-'.join(iso_pair)  
    
    for target_group in target_groups:
        if set(iso_pair.split('-')) == set(target_group.split('-')):
            return target_group
    return None 

glotlid_results['target_group'] = glotlid_results['iso_pair'].apply(
    lambda iso_pair: get_target_group(iso_pair, target_groups)
)


In [308]:
import pandas as pd

target_groups = [
    'eng_Latn-tam_Taml', 'eus_Latn-spa_Latn', 'eng_Latn-hin_Deva', 'zho_Hans-eng_Latn', 'eng_Latn-spa_Latn', 'arz_Arab-eng_Latn',
    'arz_Arab-arb_Arab', 'eng_Latn-mal_Mlym', 'eng_Latn-ars_Arab', 'eng_Latn-deu_Latn', 'eng_Latn-tur_Latn', 'arb_Latn-eng_Latn', 'eng_Latn-ind_Latn'
]

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

glotlid_results['match_type'], glotlid_results['correct_lang'] = zip(*glotlid_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = glotlid_results[glotlid_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    total_count = len(target_group_rows)
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match (%)': (full_match_count / total_count) * 100,
        'Partial Match (%)': (partial_match_count / total_count) * 100,
        'No Match (%)': (no_match_count / total_count) * 100,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T

def calculate_exact_match(true_label, predicted_label):
    return 1 if set(true_label) == set(predicted_label) else 0

def calculate_hamming_loss(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    mismatches = len(true_label_set.symmetric_difference(pred_label_set))
    return mismatches / len(true_label_set) if len(true_label_set) > 0 else 0

def calculate_fpr(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    false_positives = len(pred_label_set - true_label_set)
    return false_positives / len(true_label_set) if len(true_label_set) > 0 else 0

glotlid_results['exact_match'] = glotlid_results.apply(
    lambda row: calculate_exact_match(row['iso_pair'], row['predicted_top_2']), axis=1)

glotlid_results['hamming_loss'] = glotlid_results.apply(
    lambda row: calculate_hamming_loss(row['iso_pair'], row['predicted_top_2']), axis=1)

glotlid_results['fpr'] = glotlid_results.apply(
    lambda row: calculate_fpr(row['iso_pair'], row['predicted_top_2']), axis=1)

grouped_results = glotlid_results.groupby('target_group').agg(
    exact_match=('exact_match', 'mean'),
    hamming_loss=('hamming_loss', 'mean'),
    fpr=('fpr', 'mean'),
).reset_index()

grouped_results_glotlid = grouped_results.merge(results_df, left_on='target_group', right_index=True)

print(grouped_results_glotlid)


         target_group  exact_match  hamming_loss       fpr Full Match (%)  \
0   arb_Latn-eng_Latn     0.006309      1.599369  0.799685       0.630915   
1   arz_Arab-arb_Arab     0.049296      1.156387  0.578193       4.929577   
2   arz_Arab-eng_Latn     0.010884      1.100000  0.550000       1.088435   
3   eng_Latn-ars_Arab     0.004684      1.255269  0.627635       0.468384   
4   eng_Latn-deu_Latn     0.063241      0.948617  0.474308       6.324111   
5   eng_Latn-hin_Deva     0.000000      1.660529  0.830265            0.0   
6   eng_Latn-ind_Latn     0.100806      1.020161  0.510081      10.080645   
7   eng_Latn-mal_Mlym     0.002924      1.846784  0.923392       0.292398   
8   eng_Latn-spa_Latn     0.037369      1.062528  0.531264       3.736943   
9   eng_Latn-tam_Taml     0.000468      1.701874  0.850937       0.046838   
10  eng_Latn-tur_Latn     0.042373      1.000000  0.500000       4.237288   
11  eus_Latn-spa_Latn     0.118932      0.924757  0.462379      11.893204   

## LangDetect

In [309]:
langdetect_results = pd.read_csv("/usr/local/data/zkamel/COMP598_Project/dataset/code-switch/codeswitch_baselines_results/codeswitch/langdetect_multi_cs.csv")

In [310]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996
...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999


In [311]:
language_to_iso_script = {
    'Arabizi': 'arb',
    'Basque': 'eu',
    'Chinese': 'zh', 
    'Egyptian': 'eg',
    'English': 'en',
    'German': 'de',
    'Hindi': 'hi',
    'Indonesian': 'in',
    'MSA': 'ar',       
    'Malayalam': 'ml',
    'Saudi': 'ars',
    'Spanish': 'es',
    'Tamil': 'ta',
    'Turkish': 'tr'
}

In [312]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [313]:
langdetect_results['iso_pair'] = langdetect_results['languages'].apply(map_to_iso_pair)

In [314]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]"
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]"
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]"
...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]"


In [315]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [316]:
langdetect_results["top_pred"][0]

'[tr:0.9999969491798868]'

In [317]:
import re

langdetect_results["predicted_top_2"] = langdetect_results["top_pred"].apply(
    lambda s: re.findall(r'(\w+):', s)[:2] if isinstance(s, str) else []
)


In [318]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]",[tr]
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]","[id, tr]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]",[en]
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]",[et]
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]",[en]
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]",[en]
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]","[no, et]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]",[cn]
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]",[ar]


In [319]:
langdetect_results["predicted_top_2"] = langdetect_results["predicted_top_2"].apply(
    lambda x: x + ["UND"] if isinstance(x, list) and len(x) == 1 else x
)


In [320]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]","[tr, UND]"
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]","[id, tr]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]","[en, UND]"
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]","[et, UND]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]","[en, UND]"
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]","[en, UND]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]","[no, et]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]","[cn, UND]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]","[ar, UND]"


In [321]:
def convert_to_iso_pair(language_pair):
    lang1, lang2 = language_pair.split('-')
    iso1 = language_to_iso_script[lang1]
    iso2 = language_to_iso_script[lang2]
    iso_pair = f"{iso1.split('_')[0]}-{iso2.split('_')[0]}"
    return iso_pair

languages = ['English-Tamil', 'Basque-Spanish', 'English-Hindi',
             'Chinese-English', 'English-Spanish', 'Egyptian-English',
             'Egyptian-MSA', 'English-Malayalam', 'English-Saudi',
             'English-German', 'English-Turkish', 'Arabizi-English',
             'English-Indonesian']

iso_pairs = [convert_to_iso_pair(pair) for pair in languages]

print(iso_pairs)

['en-ta', 'eu-es', 'en-hi', 'zh-en', 'en-es', 'eg-en', 'eg-ar', 'en-ml', 'en-ars', 'en-de', 'en-tr', 'arb-en', 'en-in']


In [322]:
import pandas as pd

target_groups = ['en-ta', 'eu-es', 'en-hi', 'zh-en', 'en-es', 'eg-en', 'eg-ar', 'en-ml', 'en-ars', 'en-de', 'en-tr', 'arb-en', 'en-in']

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

langdetect_results['match_type'], langdetect_results['correct_lang'] = zip(*langdetect_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = langdetect_results[langdetect_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match': full_match_count,
        'Partial Match': partial_match_count,
        'No Match': no_match_count,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T
print(results_df)


       Full Match Partial Match No Match  \
en-ta           2           884     3384   
eu-es           0           136      276   
en-hi           1          4480     2960   
zh-en           0           773     5790   
en-es         440          5474      883   
eg-en           0           366     2574   
eg-ar           0          3910      208   
en-ml           1           267     1442   
en-ars          0            48      379   
en-de          13           240        0   
en-tr           3           113        2   
arb-en          0           145      172   
en-in           0            59      189   

       Most Frequent Correct Language (Partial)  
en-ta                                        en  
eu-es                                        es  
en-hi                                        en  
zh-en                                        en  
en-es                                        es  
eg-en                                        en  
eg-ar                            

In [323]:
def get_target_group(iso_pair, target_groups):
    if isinstance(iso_pair, list):
        iso_pair = '-'.join(iso_pair)  
    
    for target_group in target_groups:
        if set(iso_pair.split('-')) == set(target_group.split('-')):
            return target_group
    return None 

langdetect_results['target_group'] = langdetect_results['iso_pair'].apply(
    lambda iso_pair: get_target_group(iso_pair, target_groups)
)


In [324]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2,match_type,correct_lang,target_group
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]","[tr, UND]",No Match,None,en-ta
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]","[id, tr]",No Match,None,eu-es
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]","[en, UND]",Partial Match,en,en-hi
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]","[et, UND]",No Match,None,en-ta
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]","[en, UND]",Partial Match,en,zh-en
...,...,...,...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]","[en, UND]",Partial Match,en,en-hi
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]","[no, et]",No Match,None,en-ta
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]","[cn, UND]",No Match,None,zh-en
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]","[ar, UND]",No Match,None,eg-en


In [ ]:

import pandas as pd

def check_label_match(row, target_groups):
    true_label_set = set(row['iso_pair'])
    pred_label_set = set(row['predicted_top_2'])
    
    if true_label_set == pred_label_set:
        return 'Full Match', None
    
    intersection = true_label_set.intersection(pred_label_set)
    if intersection:
        correct_lang = intersection.pop() 
        return 'Partial Match', correct_lang
    
    return 'No Match', None

langdetect_results['match_type'], langdetect_results['correct_lang'] = zip(*langdetect_results.apply(lambda row: check_label_match(row, target_groups), axis=1))

results = {}

for target_group in target_groups:
    target_group_rows = langdetect_results[langdetect_results['iso_pair'].apply(lambda x: set(x) == set(target_group.split('-')))]
    
    full_match_count = sum(target_group_rows['match_type'] == 'Full Match')
    partial_match_count = sum(target_group_rows['match_type'] == 'Partial Match')
    no_match_count = sum(target_group_rows['match_type'] == 'No Match')
    total_count = len(target_group_rows)
    
    correct_languages = target_group_rows[target_group_rows['match_type'] == 'Partial Match']['correct_lang']
    most_frequent_correct_lang = correct_languages.mode().iloc[0] if not correct_languages.empty else None
    
    results[target_group] = {
        'Full Match (%)': (full_match_count / total_count) * 100,
        'Partial Match (%)': (partial_match_count / total_count) * 100,
        'No Match (%)': (no_match_count / total_count) * 100,
        'Most Frequent Correct Language (Partial)': most_frequent_correct_lang
    }

results_df = pd.DataFrame(results).T

def calculate_exact_match(true_label, predicted_label):
    return 1 if set(true_label) == set(predicted_label) else 0

def calculate_hamming_loss(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    mismatches = len(true_label_set.symmetric_difference(pred_label_set))
    return mismatches / len(true_label_set) if len(true_label_set) > 0 else 0

def calculate_fpr(true_label, predicted_label):
    true_label_set = set(true_label)
    pred_label_set = set(predicted_label)
    false_positives = len(pred_label_set - true_label_set)
    return false_positives / len(true_label_set) if len(true_label_set) > 0 else 0

langdetect_results['exact_match'] = langdetect_results.apply(
    lambda row: calculate_exact_match(row['iso_pair'], row['predicted_top_2']), axis=1)

langdetect_results['hamming_loss'] = langdetect_results.apply(
    lambda row: calculate_hamming_loss(row['iso_pair'], row['predicted_top_2']), axis=1)

langdetect_results['fpr'] = langdetect_results.apply(
    lambda row: calculate_fpr(row['iso_pair'], row['predicted_top_2']), axis=1)

grouped_results = langdetect_results.groupby('target_group').agg(
    exact_match=('exact_match', 'mean'),
    hamming_loss=('hamming_loss', 'mean'),
    fpr=('fpr', 'mean'),
).reset_index()

grouped_results_langdetect = grouped_results.merge(results_df, left_on='target_group', right_index=True)

grouped_results_langdetect["languages"] = langdetect_results["languages"]

print(grouped_results_langdetect)


   target_group  exact_match  hamming_loss       fpr Full Match (%)  \
0        arb-en     0.000000      1.542587  0.771293            0.0   
1         eg-ar     0.000000      1.038368  0.513113            0.0   
2         eg-en     0.000000      1.875510  0.937755            0.0   
3        en-ars     0.000000      1.887588  0.943794            0.0   
4         en-de     0.051383      0.948617  0.474308        5.13834   
5         en-es     0.064734      1.065176  0.532588       6.473444   
6         en-hi     0.000134      1.397662  0.698831       0.013439   
7         en-in     0.000000      1.762097  0.881048            0.0   
8         en-ml     0.000585      1.842105  0.920760        0.05848   
9         en-ta     0.000468      1.792037  0.896019       0.046838   
10        en-tr     0.025424      0.991525  0.495763       2.542373   
11        eu-es     0.000000      1.669903  0.834951            0.0   
12        zh-en     0.000000      1.882218  0.941109            0.0   

   Pa

In [326]:
grouped_results_langdetect

,target_group,exact_match,hamming_loss,fpr,Full Match (%),Partial Match (%),No Match (%),Most Frequent Correct Language (Partial)
0,arb-en,0.000000,1.542587,0.771293,0.0,45.741325,54.258675,en
1,eg-ar,0.000000,1.038368,0.513113,0.0,94.949004,5.050996,ar
2,eg-en,0.000000,1.875510,0.937755,0.0,12.44898,87.55102,en
3,en-ars,0.000000,1.887588,0.943794,0.0,11.241218,88.758782,en
4,en-de,0.051383,0.948617,0.474308,5.13834,94.86166,0.0,de
5,en-es,0.064734,1.065176,0.532588,6.473444,80.53553,12.991025,es
6,en-hi,0.000134,1.397662,0.698831,0.013439,60.206961,39.7796,en
7,en-in,0.000000,1.762097,0.881048,0.0,23.790323,76.209677,en
8,en-ml,0.000585,1.842105,0.920760,0.05848,15.614035,84.327485,en
9,en-ta,0.000468,1.792037,0.896019,0.046838,20.702576,79.250585,en


In [331]:
grouped_results_langdetect['corresponding_label'] = grouped_results_langdetect['target_group'].apply(
    lambda x: langdetect_results[langdetect_results['target_group'] == x]['languages'].values[0] if not langdetect_results[langdetect_results['target_group'] == x].empty else None
)

In [332]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2,match_type,correct_lang,target_group,exact_match,hamming_loss,fpr
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]","[tr, UND]",No Match,None,en-ta,0,2.0,1.0
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]","[id, tr]",No Match,None,eu-es,0,2.0,1.0
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]","[en, UND]",Partial Match,en,en-hi,0,1.0,0.5
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]","[et, UND]",No Match,None,en-ta,0,2.0,1.0
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]","[en, UND]",Partial Match,en,zh-en,0,1.0,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]","[en, UND]",Partial Match,en,en-hi,0,1.0,0.5
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]","[no, et]",No Match,None,en-ta,0,2.0,1.0
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]","[cn, UND]",No Match,None,zh-en,0,2.0,1.0
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]","[ar, UND]",No Match,None,eg-en,0,2.0,1.0


In [327]:
grouped_results_openlid

,target_group,exact_match,hamming_loss,fpr,Full Match (%),Partial Match (%),No Match (%),Most Frequent Correct Language (Partial)
0,arb_Latn-eng_Latn,0.000000,1.779180,0.889590,0.0,22.082019,77.917981,eng_Latn
1,arz_Arab-arb_Arab,0.023312,1.157601,0.578800,2.331229,79.577465,18.091306,arz_Arab
2,arz_Arab-eng_Latn,0.022449,1.068027,0.534014,2.244898,88.707483,9.047619,arz_Arab
3,eng_Latn-ars_Arab,0.016393,1.180328,0.590164,1.639344,78.688525,19.672131,ars_Arab
4,eng_Latn-deu_Latn,0.043478,0.960474,0.480237,4.347826,95.256917,0.395257,deu_Latn
5,eng_Latn-hin_Deva,0.000000,1.650988,0.825494,0.0,34.901223,65.098777,eng_Latn
6,eng_Latn-ind_Latn,0.092742,1.016129,0.508065,9.274194,79.83871,10.887097,ind_Latn
7,eng_Latn-mal_Mlym,0.002924,1.845614,0.922807,0.292398,14.853801,84.853801,eng_Latn
8,eng_Latn-spa_Latn,0.006326,1.297926,0.648963,0.632632,68.94218,30.425188,spa_Latn
9,eng_Latn-tam_Taml,0.000234,1.733724,0.866862,0.023419,26.580796,73.395785,eng_Latn


In [333]:
grouped_results_openlid['corresponding_label'] = grouped_results_openlid['target_group'].apply(
    lambda x: openlid_results[openlid_results['target_group'] == x]['languages'].values[0] if not openlid_results[openlid_results['target_group'] == x].empty else None
)

In [328]:
grouped_results_franc

,target_group,exact_match,hamming_loss,fpr,Full Match (%),Partial Match (%),No Match (%),Most Frequent Correct Language (Partial)
0,arb-eng,0.000000,1.735016,0.865931,0.0,26.182965,73.817035,eng
1,arz-arb,0.000000,1.149466,0.539947,0.0,78.096163,21.903837,arb
2,arz-eng,0.000000,1.900680,0.939116,0.0,7.687075,92.312925,eng
3,eng-ars,0.000000,1.937939,0.968384,0.0,6.088993,93.911007,eng
4,eng-deu,0.011858,1.011858,0.505929,1.185771,96.442688,2.371542,deu
5,eng-hin,0.000000,1.728128,0.864064,0.0,27.187206,72.812794,eng
6,eng-ind,0.008065,1.483871,0.741935,0.806452,50.0,49.193548,ind
7,eng-mal,0.000000,1.974269,0.985965,0.0,2.339181,97.660819,eng
8,eng-spa,0.001324,1.516919,0.758055,0.132411,47.962336,51.905252,spa
9,eng-tam,0.000000,1.970726,0.985129,0.0,2.880562,97.119438,eng


In [334]:
grouped_results_franc['corresponding_label'] = grouped_results_franc['target_group'].apply(
    lambda x: franc_results[franc_results['target_group'] == x]['languages'].values[0] if not franc_results[franc_results['target_group'] == x].empty else None
)

In [336]:
grouped_results_glotlid['corresponding_label'] = grouped_results_glotlid['target_group'].apply(
    lambda x: glotlid_results[glotlid_results['target_group'] == x]['languages'].values[0] if not glotlid_results[glotlid_results['target_group'] == x].empty else None
)

In [337]:
grouped_results_glotlid

,target_group,exact_match,hamming_loss,fpr,Full Match (%),Partial Match (%),No Match (%),Most Frequent Correct Language (Partial),corresponding_label
0,arb_Latn-eng_Latn,0.006309,1.599369,0.799685,0.630915,38.801262,60.567823,eng_Latn,Arabizi-English
1,arz_Arab-arb_Arab,0.049296,1.156387,0.578193,4.929577,74.502186,20.568237,arz_Arab,Egyptian-MSA
2,arz_Arab-eng_Latn,0.010884,1.100000,0.550000,1.088435,87.823129,11.088435,arz_Arab,Egyptian-English
3,eng_Latn-ars_Arab,0.004684,1.255269,0.627635,0.468384,73.5363,25.995316,ars_Arab,English-Saudi
4,eng_Latn-deu_Latn,0.063241,0.948617,0.474308,6.324111,92.490119,1.185771,deu_Latn,English-German
5,eng_Latn-hin_Deva,0.000000,1.660529,0.830265,0.0,33.94705,66.05295,eng_Latn,English-Hindi
6,eng_Latn-ind_Latn,0.100806,1.020161,0.510081,10.080645,77.822581,12.096774,ind_Latn,English-Indonesian
7,eng_Latn-mal_Mlym,0.002924,1.846784,0.923392,0.292398,14.736842,84.97076,eng_Latn,English-Malayalam
8,eng_Latn-spa_Latn,0.037369,1.062528,0.531264,3.736943,86.273356,9.989701,spa_Latn,English-Spanish
9,eng_Latn-tam_Taml,0.000468,1.701874,0.850937,0.046838,29.71897,70.234192,eng_Latn,English-Tamil


In [361]:
import pandas as pd

dataframes = [grouped_results_langdetect, grouped_results_openlid, grouped_results_franc, grouped_results_glotlid]
df_names = ['LangDetect', 'OpenLID', 'Franc', 'GlotLID']

metrics_to_aggregate = ['exact_match', 'fpr', 'Full Match (%)', 'Partial Match (%)', 'No Match (%)']
partial_match_column = 'Most Frequent Correct Language (Partial)'

processed_data = []

for df, name in zip(dataframes, df_names):
    df_cleaned = df.drop(columns=['hamming_loss'])
    df_cleaned['source'] = name
    processed_data.append(df_cleaned)

combined_df = pd.concat(processed_data, ignore_index=True)

final_df = combined_df.groupby(['corresponding_label', 'source'])[metrics_to_aggregate].mean()
final_df[partial_match_column] = combined_df.groupby(['corresponding_label', 'source'])[partial_match_column].agg(lambda x: x.mode().iloc[0] if not x.empty else None).reset_index(drop=True)

final_df = final_df.reset_index()

metric_dfs = {}
for metric in metrics_to_aggregate:
    metric_dfs[metric] = final_df.pivot_table(index='corresponding_label', columns='source', values=metric)

partial_match_with_lang_df = final_df.pivot_table(index='corresponding_label', columns='source', values=['Partial Match (%)', partial_match_column])

print(partial_match_with_lang_df)


                    Partial Match (%)                                 
source                          Franc    GlotLID LangDetect    OpenLID
corresponding_label                                                   
Arabizi-English             26.182965  38.801262  45.741325  22.082019
Basque-Spanish              53.883495  83.737864  33.009709  81.067961
Chinese-English              3.885418   5.942404   11.77815  96.663111
Egyptian-English             7.687075  87.823129   12.44898  88.707483
Egyptian-MSA                78.096163  74.502186  94.949004  79.577465
English-German              96.442688  92.490119   94.86166  95.256917
English-Hindi               27.187206   33.94705  60.206961  34.901223
English-Indonesian               50.0  77.822581  23.790323   79.83871
English-Malayalam            2.339181  14.736842  15.614035  14.853801
English-Saudi                6.088993    73.5363  11.241218  78.688525
English-Spanish             47.962336  86.273356   80.53553   68.94218
Englis

In [360]:
metric_dfs.keys()

dict_keys(['exact_match', 'fpr', 'Full Match (%)', 'Partial Match (%)', 'No Match (%)'])

In [365]:
metric_dfs['Partial Match (%)']

source,Franc,GlotLID,LangDetect,OpenLID
corresponding_label,,,,
Arabizi-English,26.182965,38.801262,45.741325,22.082019
Basque-Spanish,53.883495,83.737864,33.009709,81.067961
Chinese-English,3.885418,5.942404,11.77815,96.663111
Egyptian-English,7.687075,87.823129,12.44898,88.707483
Egyptian-MSA,78.096163,74.502186,94.949004,79.577465
English-German,96.442688,92.490119,94.86166,95.256917
English-Hindi,27.187206,33.94705,60.206961,34.901223
English-Indonesian,50.0,77.822581,23.790323,79.83871
English-Malayalam,2.339181,14.736842,15.614035,14.853801


In [ ]:
dataframes = [grouped_results_langdetect, grouped_results_openlid, grouped_results_franc, grouped_results_glotlid]
df_names = ['LangDetect', 'OpenLID', 'Franc', 'GlotLID']
most_frequent_languages = {}

for df, name in zip(dataframes, df_names):
    most_frequent = df.groupby('corresponding_label')['Most Frequent Correct Language (Partial)'].agg(lambda x: x.mode().iloc[0] if not x.empty else None)
    most_frequent_df = most_frequent.reset_index(name=f'Most Frequent Correct Language (Partial) ({name})')
    most_frequent_languages[name] = most_frequent_df

final_table = most_frequent_languages['LangDetect']
for name in df_names[1:]:
    final_table = pd.merge(final_table, most_frequent_languages[name], on='corresponding_label', how='left')

print(final_table)


   corresponding_label Most Frequent Correct Language (Partial) (LangDetect)  \
0      Arabizi-English                                                 en      
1       Basque-Spanish                                                 es      
2      Chinese-English                                                 en      
3     Egyptian-English                                                 en      
4         Egyptian-MSA                                                 ar      
5       English-German                                                 de      
6        English-Hindi                                                 en      
7   English-Indonesian                                                 en      
8    English-Malayalam                                                 en      
9        English-Saudi                                                 en      
10     English-Spanish                                                 es      
11       English-Tamil                  